In [2]:
%pip install --upgrade langchain
%pip install -U langchain-community

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import redis
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.redis import RedisSaver
from langchain_ollama import ChatOllama
from langchain_core.messages.utils import (
    count_tokens_approximately,
    trim_messages)
import hashlib
import json
#from langchain.globals import set_llm_cache
from langchain_community.cache import RedisCache

/tmp/ipykernel_108/623163877.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.cache import RedisCache


In [4]:
from langchain_core.globals import set_llm_cache

In [5]:
# 1. Define the Graph State                                                                                                                                                                          
class State(TypedDict):
    # add_messages appends new messages to the existing list instead of overwriting                                                                                                                  
    messages: Annotated[list, add_messages]


In [6]:
from typing import Any, List, Optional
from langchain_core.caches import BaseCache
from langchain_core.outputs import Generation
import hashlib

In [7]:
class RedisTTLCache(BaseCache):

    def __init__(self,
                 redis_client: redis.Redis,
                 prefix: str = 'llm_cache:',
                 ttl_seconds: int = 86400):
        self.redis = redis_client
        self.prefix = prefix
        self.ttl = ttl_seconds

    def _hash(self, prompt: str, llm_string: str)-> str:
        val = f"{prompt}:{llm_string}"
        return hashlib.sha256(val.encode()).hexdigest()
        
    def _key(self, prompt: str, llm_string: str) -> str:
        base_key = self._hash(prompt, llm_string)
        key = f"{self.prefix}{base_key}"
        print(f"[CACHE KEY]: {key}")
        return key
        
    def lookup(self, prompt: str, llm_string: str):
        key = self._key(prompt, llm_string)
        result = self.redis.get(key)
        if result:
            print(f"[CACHE HIT]: {key}")
        else:
            print(f"[CACHE MISS]: {key}")
        return super().lookup(prompt, llm_string) if result else None

    def update(self, prompt: str, llm_string: str, return_val: List[Generation]) -> None:
        key = self._key(prompt, llm_string)
        print(f"[CACHE SET]: {key} TTL={self.ttl}s")
        self.redis.setex(key, self.ttl, json.dumps([g.dict() for g in return_val]))

    def clear(self) -> None:
        cursor = 0
        pattern = f"{self.prefix}"
        while True:
            cursor, keys = self.redis.scan(cursor=cursor, match=pattern, count=500)
            if keys:
                self.redis.delete(*keys)
            if cursor == 0:
                break
        

In [19]:
class Chatbot():
    def __init__(self):
        self.user_id = 'user_id:test123'
        self.model = 'llama3.2'
        self.temperature = 0.7
        self.host = 'redis'
        self.port = 6379
        self.prefix= 'llama_app:'
        self.redis_llm_client = None
        self.checkpointer = None
        self.llm_cache = None

    # Define the node function that calls Ollama
    def call_model(self, state: State):
    # Initialize the Ollama model via LangChain
        model = ChatOllama(model=self.model, temperature=self.temperature)
        trimmed_messages = trim_messages(state["messages"],
                                         strategy="last",  # Keep the most recent messages
                                         token_counter=count_tokens_approximately,  # Count tokens locally
                                         max_tokens=400,  # Maximum token limit for the LLM input
                                         start_on="human",  # Ensure history starts with a human message
                                         include_system=True)
        response = model.invoke(trimmed_messages)
        return {"messages": [response]}

    # Define redis checkpointer
    def redis_checkpointer(self):
        #redis_client = SafeRedisCache(host='redis', port=6379, prefix='llama_app:')
        redis_client = redis.Redis(host=self.host, port=self.port, db=0, decode_responses=False)
        ttl_config = {"default_ttl": 1440,     # Expire checkpoints after 1440 minutes (24 Hours)
                      "refresh_on_read": True   #Reset the expiration clock whenever the thread is read or updated
                     }
        checkp = RedisSaver(redis_client=redis_client,
                            ttl=ttl_config)
        checkp.setup()
        self.redis_llm_client = redis_client
        self.checkpointer = checkp
        return checkp

    def initiate_llm_cache(self):
        #redis_client = SafeRedisCache(host='redis', port=6379, prefix='llm-cache:', 
        #                              db=1, decode_responses=True)
        redis_client = redis.Redis(host=self.host, port = self.port,
                                   db=1, decode_responses=True)
        redisttlcache = RedisTTLCache(redis_client, "test:", 20)
        self.llm_cache = redisttlcache
        set_llm_cache(redisttlcache)

    def compact_thread(self, thread_id: str, keep_last: int=1):
        pattern = f"checkpoint:{thread_id}:*"
        keys = sorted(self.redis_llm_client.keys(pattern))
        to_delete = keys[:-keep_last]
        if to_delete:
            self.redis_llm_client.delete(*to_delete)
            print(f"Compacted {thread_id} : delete {len(to_delete)} old checkpoints")
            

    def compile_graph(self):
        workflow = StateGraph(State)
    
        # Add the chatbot node
        workflow.add_node("chatbot", self.call_model)
    
        # Configure entry and exit points
        workflow.add_edge(START, "chatbot")
        workflow.add_edge("chatbot", END)
        checkpointer = self.redis_checkpointer()
        return workflow.compile(checkpointer=checkpointer)




In [20]:
def main_app():
    # Create a configuration dictionary defining this unique thread session                                                                                                                               
    thread_config = {"configurable": {"thread_id": "session_abc_123", 
                                      "checkpoint_ns": "v1:customer_support_agent"}}
    chatb = Chatbot()
    
    graph = chatb.compile_graph()

    chatb.initiate_llm_cache()
    # --- Interaction Turn 1 ---                                                                                                                                                                          
    print("--- Turn 1 ---")
    input_1 = {"messages": [{"role": "user", "content": "Hi, my name is Alex and I love building AI agents."}]}
    output_1 = graph.invoke(input_1, config=thread_config)
    print(f"Bot: {output_1['messages'][-1].content}\n")

    # --- Interaction Turn 2 (Remembers Context) ---                                                                                                                                                      
    print("--- Turn 2 ---")
    input_2 = {"messages": [{"role": "user", "content": "What is my name and what do I like?"}]}
    output_2 = graph.invoke(input_2, config=thread_config)
    print(f"Bot: {output_2['messages'][-1].content}\n")

    chatb.compact_thread('session_abc_123', 5)
    chatb.checkpointer.delete_thread(thread_config)
    chatb.llm_cache.clear()

In [21]:
# 4. Test the Conversation Thread                                                                                                                                                                         
if __name__ == "__main__":
    main_app()


--- Turn 1 ---
[CACHE KEY]: test:d5c882fa943b16f90b75671e23cfd56ff4bdcb7d742a8e5c5efc876da7dd0fd1
[CACHE MISS]: test:d5c882fa943b16f90b75671e23cfd56ff4bdcb7d742a8e5c5efc876da7dd0fd1
[CACHE KEY]: test:d5c882fa943b16f90b75671e23cfd56ff4bdcb7d742a8e5c5efc876da7dd0fd1
[CACHE SET]: test:d5c882fa943b16f90b75671e23cfd56ff4bdcb7d742a8e5c5efc876da7dd0fd1 TTL=20s
Bot: Hi Alex! It's great to chat with you about your passion for building AI agents. Is there anything specific you'd like to talk about, ask, or share with me? I'm all ears!

--- Turn 2 ---
[CACHE KEY]: test:e49d477cdab50b4c7329a878ef18b13c0afc4a305b0c57f41d4dcf6069a4e2c4
[CACHE MISS]: test:e49d477cdab50b4c7329a878ef18b13c0afc4a305b0c57f41d4dcf6069a4e2c4


/tmp/ipykernel_108/2970222971.py:33: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  self.redis.setex(key, self.ttl, json.dumps([g.dict() for g in return_val]))
/tmp/ipykernel_108/2970222971.py:33: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  self.redis.setex(key, self.ttl, json.dumps([g.dict() for g in return_val]))


[CACHE KEY]: test:e49d477cdab50b4c7329a878ef18b13c0afc4a305b0c57f41d4dcf6069a4e2c4
[CACHE SET]: test:e49d477cdab50b4c7329a878ef18b13c0afc4a305b0c57f41d4dcf6069a4e2c4 TTL=20s
Bot: Your name is Alex, and you like building AI agents.

Compacted session_abc_123 : delete 6 old checkpoints


/tmp/ipykernel_108/2970222971.py:33: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  self.redis.setex(key, self.ttl, json.dumps([g.dict() for g in return_val]))
/tmp/ipykernel_108/2970222971.py:33: DeprecationWarning: Call to deprecated setex. (Use 'set' instead.) -- Deprecated since version 2.6.12.
  self.redis.setex(key, self.ttl, json.dumps([g.dict() for g in return_val]))
